In [1]:
from configuracoes_notebooks import set_proj_dir
set_proj_dir()

O diretorio do seu projeto é coleta_cebrap
Caminho absoluto do diretorio encontrado C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap
Caminho no path.


In [2]:
import geopandas as gpd
import os

from notebooks.jupyter import utils
from utils import get_data_diretorio
from utils.downloads import download_malha_geosampa

# Profundidade, Elevação, Cota  

# Área de Inundação no Território / Total de área do distrito

In [3]:
data_path= get_data_diretorio()
assets_path = os.path.join(
    data_path,
    'assets'
)

# Dependências

Este notebook é dependente dos parquets resultantes dos notebooks "overlay_mancha_inund_distrit" e "../../arborizacao_viaria/malha_distritos"

In [4]:
gdf_distrito = gpd.read_parquet(
    os.path.join(
        data_path,
        'assets',
        'distrito_ibge.parquet'
    )
)

In [5]:
gdf_overlay_inund = gpd.read_parquet(
    os.path.join(
        assets_path,
        'areas_inundacao',
        'overlay_mancha_inund_dist.parquet'
    )
)

Como os dados de Profundidade e Elevação e Cota estão disponíveis, seria possível calcular o volume de inundação, segue exemplo: 

# Cálculo dos volumes de Inundação

In [6]:
gdf_overlay_inund.sample(2)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,area_mancha_inund,CD_MUN,NM_MUN,CD_DIST,...,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,area_inund_recort
138862,334074,Jaguaré,86.6,742.864,743.03,0.17,86.6,3550308,São Paulo,355030865,...,São Paulo,350001,São Paulo,3550308,São Paulo/SP,12.164313,117738,49006,"POLYGON ((319209.756 7390702.34, 319204.756 73...",86.6
29484,227582,Água Vermelha e Lajeado,86.6,735.114,735.13,0.02,86.6,3550308,São Paulo,355030844,...,São Paulo,350001,São Paulo,3550308,São Paulo/SP,9.133494,129409,48496,"POLYGON ((354913.024 7401847.66, 354913.024 74...",86.6


In [7]:
gdf_overlay_inund.columns

Index(['cd_mancha_inund', 'nm_bacia_h', 'qt_area_me', 'qt_elevaca',
       'qt_cota_in', 'qt_profund', 'area_mancha_inund', 'CD_MUN', 'NM_MUN',
       'CD_DIST', 'NM_DIST', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI',
       'CD_CONCURB', 'NM_CONCURB', 'AREA_KM2', 'total_pop', 'total_dom',
       'geometry', 'area_inund_recort'],
      dtype='object')

In [8]:
gdf_overlay_inund['volume_inund'] = (
    gdf_overlay_inund['qt_area_me']*gdf_overlay_inund['qt_profund']
)

Para saber a profundidade CORRETA, considerando a área a partir da geometria, precisaríamos fazer uma regra de 3:
$$\text{Profundidade "correta"}=\frac{\text{Profundidade cadastrada}*\text{Área "correta"}}{\text{Área cadastrada}}$$


In [9]:
gdf_overlay_inund.sample(3)

,cd_mancha_inund,nm_bacia_h,qt_area_me,qt_elevaca,qt_cota_in,qt_profund,area_mancha_inund,CD_MUN,NM_MUN,CD_DIST,...,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,area_inund_recort,volume_inund
167797,361435,Morro do S,21.65,737.501,740.47,2.97,21.65,3550308,São Paulo,355030846,...,350001,São Paulo,3550308,São Paulo/SP,25.911404,259377,105756,"POLYGON ((321707.202 7384133.565, 321704.702 7...",21.65,64.3005
28202,226300,Água Vermelha e Lajeado,86.60,735.403,735.44,0.04,86.60,3550308,São Paulo,355030844,...,350001,São Paulo,3550308,São Paulo/SP,9.133494,129409,48496,"POLYGON ((355158.024 7401769.72, 355158.024 74...",86.60,3.4640
11244,209640,Lapa,86.60,720.971,721.71,0.74,86.60,3550308,São Paulo,355030848,...,350001,São Paulo,3550308,São Paulo/SP,10.275987,75533,36633,"POLYGON ((326996.629 7398346.508, 326991.629 7...",86.60,64.0840
